<a href="https://colab.research.google.com/github/mfaiz23/Churn-Prediction-OCTE/blob/main/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📦 NOTEBOOK 01 — PRE-PROCESSING
**Eksperimen 4: SMOTE Before Split + OCTE (Optimal Causal Decision Trees Ensemble)**

---
## Deskripsi
Notebook ini adalah **tahap pertama** dari 4 notebook yang membentuk alur penelitian eksperimen. Tujuannya adalah mempersiapkan data mentah menjadi dataset bersih yang siap untuk tahap selanjutnya.

## Alur Notebook Ini:
1. Mount Google Drive & Setup Direktori
2. Install & Import Library
3. Load Dataset
4. Eliminasi Kolom Noise & Leakage
5. Enforcing Data Types
6. Definisi Variabel Treatment (Kausal)
7. Encoding Variabel Kategorikal
8. Export Data Bersih

> ⚠️ **Jalankan cell dari atas ke bawah secara berurutan.**

---
## Cell 1 — Mount Google Drive
Mount Google Drive agar semua data dan artefak eksperimen dapat disimpan dan diakses secara persisten.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive berhasil di-mount.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive berhasil di-mount.


---
## Cell 2 — Setup Struktur Direktori Eksperimen
Membuat struktur folder yang konsisten untuk menyimpan data, model, gambar, dan hasil eksperimen.

In [ ]:
import os

BASE_PATH = '/content/drive/MyDrive/last_experiment'

folders = [
    'data/raw',
    'data/processed',
    'notebooks',
    'figures/smote',
    'figures/features',
    'figures/classification',
    'figures/causal',
    'figures/insights',
    'results/metrics',
    'results/models',
    'results/predictions',
    'references'
]

for folder in folders:
    os.makedirs(os.path.join(BASE_PATH, folder), exist_ok=True)

print('✅ Struktur direktori eksperimen berhasil dibangun!')
print(f'📁 Base path: {BASE_PATH}')

✅ Struktur direktori eksperimen berhasil dibangun!
📁 Base path: /content/drive/MyDrive/last_experiment


---
## Cell 3 — Import Library
Import semua library yang dibutuhkan untuk tahap pre-processing.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('✅ Library berhasil di-import.')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')

✅ Library berhasil di-import.
   pandas  : 2.2.2
   numpy   : 2.0.2


---
## Cell 4 — Load Dataset
Memuat dataset mentah dari Google Drive. Dataset yang digunakan adalah **Telco Customer Churn** dari IBM.

> 📌 **Pastikan file `Telco_customer_churn.xlsx` sudah diupload ke folder `data/raw/` di Google Drive sebelum menjalankan cell ini.**

In [ ]:
FILE_PATH = f'{BASE_PATH}/data/raw/Telco_customer_churn.xlsx'

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f'❌ File tidak ditemukan di {FILE_PATH}.\n'
        '   Pastikan Anda sudah mengupload dataset ke folder data/raw/ di Google Drive.'
    )

df = pd.read_excel(FILE_PATH)

# Bersihkan spasi ekstra pada nama kolom
df.columns = df.columns.str.strip()

print(f'📊 Dataset berhasil dimuat.')
print(f'   Dimensi awal : {df.shape[0]:,} baris x {df.shape[1]} kolom')
print(f'\n🔎 Preview 5 baris pertama:')
display(df.head())
print(f'\n📋 Tipe data kolom:')
display(df.dtypes.to_frame('dtype'))

📊 Dataset berhasil dimuat.
   Dimensi awal : 7,043 baris x 33 kolom

🔎 Preview 5 baris pertama:


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices



📋 Tipe data kolom:


,dtype
CustomerID,object
Count,int64
Country,object
State,object
City,object
Zip Code,int64
Lat Long,object
Latitude,float64
Longitude,float64
Gender,object


---
## Cell 5 — Eliminasi Kolom Noise & Data Leakage
Menghapus kolom yang tidak relevan secara ilmiah:
- **Identitas**: `CustomerID`, `Count`, lokasi geografis
- **Data leakage**: `Churn Label`, `Churn Score`, `CLTV`, `Churn Reason` — karena ini adalah informasi yang hanya diketahui **setelah** churn terjadi, sehingga tidak boleh menjadi fitur prediksi.

In [ ]:
# Normalisasi nama kolom menjadi lowercase untuk pencocokan yang aman
df.columns = [c.lower() for c in df.columns]

COLUMNS_TO_DROP = [
    'Customerid', 'count', 'country', 'state', 'city', 'zip code',
    'lat long', 'latitude', 'longitude', 'churn label', 'churn score',
    'cltv', 'churn reason'
]

existing_drops = [col for col in COLUMNS_TO_DROP if col in df.columns]
df = df.drop(columns=existing_drops, axis=1)

print(f'✂️  Berhasil mengeliminasi {len(existing_drops)} kolom noise/leakage:')
for col in existing_drops:
    print(f'   - {col}')
print(f'\n📊 Dimensi setelah eliminasi: {df.shape[0]:,} baris x {df.shape[1]} kolom')

✂️  Berhasil mengeliminasi 12 kolom noise/leakage:
   - count
   - country
   - state
   - city
   - zip code
   - lat long
   - latitude
   - longitude
   - churn label
   - churn score
   - cltv
   - churn reason

📊 Dimensi setelah eliminasi: 7,043 baris x 21 kolom


---
## Cell 6 — Penanganan Missing Values & Enforcing Data Types
Kolom numerik kontinu (`tenure`, `monthlycharges`, `totalcharges`) dikonversi secara paksa ke `float`. Nilai yang tidak bisa dikonversi (misal string kosong) diganti dengan 0.

> 💡 `totalcharges` sering berisi string kosong pada pelanggan baru (tenure=0) di dataset IBM Telco.

In [ ]:
# Cek missing values sebelum cleaning
missing_before = df.isnull().sum()
print('🔍 Missing values sebelum cleaning:')
display(missing_before[missing_before > 0].to_frame('missing_count'))

# Enforce numeric types
NUMERIC_COLS = ['tenure', 'monthly charges', 'total charges']
# Coba juga nama kolom alternatif (tanpa spasi)
NUMERIC_COLS_ALT = ['tenure', 'monthlycharges', 'totalcharges']

for col in NUMERIC_COLS + NUMERIC_COLS_ALT:
    if col in df.columns:
        before = df[col].dtype
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
        print(f'   ✅ {col}: {before} → {df[col].dtype}')

# Cek missing values setelah cleaning
missing_after = df.isnull().sum().sum()
print(f'\n✅ Total missing values setelah cleaning: {missing_after}')

🔍 Missing values sebelum cleaning:


,missing_count


   ✅ monthly charges: float64 → float64
   ✅ total charges: object → float64

✅ Total missing values setelah cleaning: 0


---
## Cell 7 — Definisi Variabel Treatment (Kausal)
Ini adalah langkah **inti untuk analisis kausal**. Variabel `Contract` dikodekan sebagai variabel treatment biner `treatment_w`:

- `treatment_w = 1` → Pelanggan dengan **kontrak Two Year** (diperlakukan / treated)
- `treatment_w = 0` → Pelanggan dengan kontrak Month-to-Month atau One Year (kontrol)

Interpretasi: kita ingin mengukur **efek kausal** dari memiliki kontrak jangka panjang terhadap probabilitas churn.

In [ ]:
if 'contract' not in df.columns:
    raise KeyError("Kolom 'contract' tidak ditemukan.")

df['treatment_w'] = np.where(df['contract'] == 'Two year', 1, 0)
df = df.drop(columns=['contract'], axis=1)

treatment_dist = df['treatment_w'].value_counts()
print('💊 Distribusi Variabel Treatment (treatment_w):')
print(f'   W=0 (Kontrol/Non-Two Year) : {treatment_dist.get(0, 0):,} observasi ({treatment_dist.get(0, 0)/len(df)*100:.1f}%)')
print(f'   W=1 (Treated/Two Year)     : {treatment_dist.get(1, 0):,} observasi ({treatment_dist.get(1, 0)/len(df)*100:.1f}%)')

💊 Distribusi Variabel Treatment (treatment_w):
   W=0 (Kontrol/Non-Two Year) : 5,348 observasi (75.9%)
   W=1 (Treated/Two Year)     : 1,695 observasi (24.1%)


---
## Cell 8 — Identifikasi & Encoding Variabel Kategorikal
Proses encoding dilakukan secara **selektif dan aman**:
1. Hanya kolom dengan tipe `object` yang diproses
2. Kolom dengan nilai unik > 10 di-drop otomatis untuk mencegah column explosion
3. One-Hot Encoding dengan `drop_first=True` untuk menghindari multikolinearitas

In [ ]:
TARGET_COL = 'churn value'
if TARGET_COL not in df.columns:
    raise KeyError(f"❌ Kolom target '{TARGET_COL}' tidak ditemukan.")

target = df[TARGET_COL]
features = df.drop(columns=[TARGET_COL], axis=1)

# Identifikasi kolom kategorikal
categorical_cols = []
print('🔍 Analisis Kolom Kategorikal:')
print(f'{"Kolom":<30} {"Unik":>6} {"Status"}')
print('-' * 55)

for col in features.select_dtypes(include=['object']).columns:
    unique_count = features[col].nunique()
    if unique_count < 10:
        categorical_cols.append(col)
        status = f'✅ Lolos ({unique_count} nilai unik)'
    else:
        features = features.drop(columns=[col], axis=1)
        status = f'❌ Di-drop (terlalu banyak: {unique_count})'
    print(f'{col:<30} {unique_count:>6}  {status}')

# One-Hot Encoding
features_encoded = pd.get_dummies(features, columns=categorical_cols, drop_first=True)
print(f'\n✅ Encoding selesai.')
print(f'   Jumlah fitur sebelum encoding : {len(features.columns)}')
print(f'   Jumlah fitur setelah encoding  : {len(features_encoded.columns)}')

🔍 Analisis Kolom Kategorikal:
Kolom                            Unik Status
-------------------------------------------------------
customerid                       7043  ❌ Di-drop (terlalu banyak: 7043)
gender                              2  ✅ Lolos (2 nilai unik)
senior citizen                      2  ✅ Lolos (2 nilai unik)
partner                             2  ✅ Lolos (2 nilai unik)
dependents                          2  ✅ Lolos (2 nilai unik)
phone service                       2  ✅ Lolos (2 nilai unik)
multiple lines                      3  ✅ Lolos (3 nilai unik)
internet service                    3  ✅ Lolos (3 nilai unik)
online security                     3  ✅ Lolos (3 nilai unik)
online backup                       3  ✅ Lolos (3 nilai unik)
device protection                   3  ✅ Lolos (3 nilai unik)
tech support                        3  ✅ Lolos (3 nilai unik)
streaming tv                        3  ✅ Lolos (3 nilai unik)
streaming movies                    3  ✅ Lolos (3 nil

---
## Cell 9 — Ringkasan Dataset Final & Validasi
Menampilkan ringkasan statistik akhir sebelum data diekspor.

In [ ]:
# Gabungkan fitur + target
df_final = pd.concat([features_encoded, target], axis=1)

print('📊 Ringkasan Dataset Final:')
print(f'   Total baris   : {df_final.shape[0]:,}')
print(f'   Total kolom   : {df_final.shape[1]}')
print(f'   Missing values: {df_final.isnull().sum().sum()}')

# Distribusi target
churn_dist = df_final[TARGET_COL].value_counts()
print(f'\n🎯 Distribusi Target (Churn Value):')
print(f'   Tidak Churn (0): {churn_dist.get(0, 0):,} ({churn_dist.get(0, 0)/len(df_final)*100:.1f}%)')
print(f'   Churn (1)      : {churn_dist.get(1, 0):,} ({churn_dist.get(1, 0)/len(df_final)*100:.1f}%)')
print(f'\n📋 Preview dataset final:')
display(df_final.head())
print(f'\n📈 Statistik deskriptif:')
display(df_final.describe())

📊 Ringkasan Dataset Final:
   Total baris   : 7,043
   Total kolom   : 30
   Missing values: 0

🎯 Distribusi Target (Churn Value):
   Tidak Churn (0): 5,174 (73.5%)
   Churn (1)      : 1,869 (26.5%)

📋 Preview dataset final:


,tenure months,monthly charges,total charges,treatment_w,gender_Male,senior citizen_Yes,partner_Yes,dependents_Yes,phone service_Yes,multiple lines_No phone service,...,tech support_Yes,streaming tv_No internet service,streaming tv_Yes,streaming movies_No internet service,streaming movies_Yes,paperless billing_Yes,payment method_Credit card (automatic),payment method_Electronic check,payment method_Mailed check,churn value
0,2,53.85,108.15,0,True,False,False,False,True,False,...,False,False,False,False,False,True,False,False,True,1
1,2,70.70,151.65,0,False,False,False,True,True,False,...,False,False,False,False,False,True,False,True,False,1
2,8,99.65,820.50,0,False,False,False,True,True,False,...,False,False,True,False,True,True,False,True,False,1
3,28,104.80,3046.05,0,False,False,True,True,True,False,...,True,False,True,False,True,True,False,True,False,1
4,49,103.70,5036.30,0,True,False,False,True,True,False,...,False,False,True,False,True,True,False,False,False,1



📈 Statistik deskriptif:


,tenure months,monthly charges,total charges,treatment_w,churn value
count,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,32.371149,64.761692,2279.734304,0.240664,0.265370
std,24.559481,30.090047,2266.794470,0.427517,0.441561
min,0.000000,18.250000,0.000000,0.000000,0.000000
25%,9.000000,35.500000,398.550000,0.000000,0.000000
50%,29.000000,70.350000,1394.550000,0.000000,0.000000
75%,55.000000,89.850000,3786.600000,0.000000,1.000000
max,72.000000,118.750000,8684.800000,1.000000,1.000000


---
## Cell 10 — Export Dataset Bersih ke Google Drive
Menyimpan hasil pre-processing ke Google Drive agar dapat digunakan oleh notebook selanjutnya.

In [ ]:
OUTPUT_PATH = f'{BASE_PATH}/data/processed/data_cleaned.csv'
df_final.to_csv(OUTPUT_PATH, index=False)

print('💾 Dataset bersih berhasil disimpan!')
print(f'   Path  : {OUTPUT_PATH}')
print(f'   Shape : {df_final.shape[0]:,} baris x {df_final.shape[1]} kolom')
print('\n✅ Notebook 01 selesai. Lanjutkan ke Notebook 02: SMOTE & Train-Test Split.')

💾 Dataset bersih berhasil disimpan!
   Path  : /content/drive/MyDrive/last_experiment/data/processed/data_cleaned.csv
   Shape : 7,043 baris x 30 kolom

✅ Notebook 01 selesai. Lanjutkan ke Notebook 02: SMOTE & Train-Test Split.
